# LeetCode 3902 — Zigzag Level Sum of Binary Tree bằng Python

- Link: https://leetcode.com/problems/zigzag-level-sum-of-binary-tree/
- Độ khó: **Hard** (theo danh mục) · 🔒 **Premium**
- Chủ đề: Cây nhị phân, duyệt theo tầng (BFS), DFS theo độ sâu, tổng tiền tố/hậu tố có điểm dừng

> **Ghi chú độ khó:** trên LeetCode bài này ghi **Medium**; notebook vẫn lưu ở `DSA/Hard` theo danh mục.
>
> **Ghi chú Premium:** đề bài được viết lại theo hiểu biết về bài (dựa trên bản mô tả công khai của doocs). Nếu có tài khoản Premium, nên đối chiếu lại đề gốc.

Skeleton trên LeetCode (Python3, theo doocs):

```python
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None): ...
class Solution:
    def zigzagLevelSum(self, root: Optional[TreeNode]) -> List[int]:
```

## 1. Mục tiêu học

1. Duyệt cây **theo tầng** bằng BFS (dùng danh sách tầng hiện tại / tầng kế tiếp).
2. Xử lý một tầng theo **hai chiều luân phiên** (trái → phải, phải → trái) mà không cần đảo mảng.
3. Cài đặt đúng quy tắc **dừng ngay trước nút vi phạm đầu tiên** (thiếu con trái ở tầng lẻ, thiếu con phải ở tầng chẵn).
4. Thấy góc nhìn **DFS một lượt**: tầng lẻ là *tổng tiền tố đến điểm dừng*, tầng chẵn là *tổng hậu tố sau nút vi phạm cuối cùng* (khi duyệt trái → phải).
5. Dựng cây từ biểu diễn mảng LeetCode và viết test đối chiếu nhiều cách.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho gốc `root` của cây nhị phân. Duyệt theo tầng, đánh số tầng từ 1:

- **Tầng lẻ**: duyệt các nút từ **trái sang phải**; dừng ngay trước nút đầu tiên **không có con trái**.
- **Tầng chẵn**: duyệt từ **phải sang trái**; dừng ngay trước nút đầu tiên **không có con phải**.

Chỉ những nút được duyệt **trước** điểm dừng mới cộng vào tổng của tầng. Trả về mảng `ans` với `ans[i]` là tổng tầng `i + 1` (có đủ một phần tử cho **mỗi tầng** của cây, kể cả khi tổng bằng 0).

**Ví dụ 1:** `root = [5,2,8,1,null,9,6]` → `[5,8,0]`.

- Tầng 1: nút 5 có con trái ⇒ cộng 5.
- Tầng 2 (phải → trái): 8 có con phải (6) ⇒ cộng; 2 không có con phải ⇒ dừng. Tổng 8.
- Tầng 3: nút đầu tiên (1) không có con trái ⇒ dừng ngay, tổng 0.

**Ví dụ 2:** `root = [1,2,3,4,5,null,7]` → `[1,5,0]`.

**Ràng buộc:** số nút trong `[1, 10^5]`, `-10^5 ≤ Node.val ≤ 10^5`.

In [1]:
from typing import List, Optional
from collections import deque
import random, time

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def build(arr):
    """Dựng cây từ mảng kiểu LeetCode (BFS, None là nút rỗng)."""
    if not arr or arr[0] is None:
        return None
    root = TreeNode(arr[0])
    q = deque([root]); i = 1
    while q and i < len(arr):
        node = q.popleft()
        if i < len(arr) and arr[i] is not None:
            node.left = TreeNode(arr[i]); q.append(node.left)
        i += 1
        if i < len(arr) and arr[i] is not None:
            node.right = TreeNode(arr[i]); q.append(node.right)
        i += 1
    return root

examples = [([5, 2, 8, 1, None, 9, 6], [5, 8, 0]), ([1, 2, 3, 4, 5, None, 7], [1, 5, 0])]
for arr, exp in examples:
    print(arr, "-> kỳ vọng", exp)

[5, 2, 8, 1, None, 9, 6] -> kỳ vọng [5, 8, 0]
[1, 2, 3, 4, 5, None, 7] -> kỳ vọng [1, 5, 0]


## 3. Kiến thức nền

### 3.1. Cây nhị phân và biểu diễn mảng của LeetCode

Mảng `[5,2,8,1,null,9,6]` liệt kê các nút theo tầng, trái sang phải; `null` là chỗ trống. Hàm `build` ở trên dựng lại cây bằng hàng đợi. Hàm dưới in cây theo tầng để kiểm tra.

In [2]:
def levels_of(root):
    res, cur = [], [root] if root else []
    while cur:
        res.append([n.val for n in cur])
        cur = [c for n in cur for c in (n.left, n.right) if c]
    return res

print(levels_of(build([5, 2, 8, 1, None, 9, 6])))

[[5], [2, 8], [1, 9, 6]]


### 3.2. BFS theo tầng: "tầng hiện tại" và "tầng kế tiếp"

Thay vì một `deque` chung, ta giữ list `cur` (tầng hiện tại) và dựng `nxt` (tầng sau) bằng cách duyệt `cur` từ trái sang phải và thêm con trái rồi con phải. Như vậy mỗi tầng là một list **theo thứ tự trái → phải**.

In [3]:
root = build([1, 2, 3, 4, 5, None, 7])
cur, depth = [root], 1
while cur:
    print("tầng", depth, [n.val for n in cur])
    nxt = []
    for n in cur:
        if n.left: nxt.append(n.left)
        if n.right: nxt.append(n.right)
    cur, depth = nxt, depth + 1

tầng 1 [1]
tầng 2 [2, 3]
tầng 3 [4, 5, 7]


### 3.3. Duyệt một list theo chiều ngược không cần đảo

`reversed(lst)` trả về iterator ngược, không tạo bản sao. Hoặc dùng chỉ số `lst[m - 1 - i]`.

In [4]:
lst = [10, 20, 30]
print(list(reversed(lst)), [lst[len(lst) - 1 - i] for i in range(len(lst))])

[30, 20, 10] [30, 20, 10]


### 3.4. Tổng tiền tố đến điểm dừng và `break`

"Cộng cho tới khi gặp phần tử xấu đầu tiên" là một vòng lặp có `break`. Phiên bản ngược lại — "tổng các phần tử **sau** phần tử xấu cuối cùng" — có thể tính khi duyệt xuôi bằng cách **đặt lại tổng về 0** mỗi khi gặp phần tử xấu.

In [5]:
vals = [3, 4, -1, 5, 6]          # -1 đánh dấu phần tử "xấu"
pref = 0
for v in vals:
    if v < 0: break
    pref += v
suf = 0
for v in vals:
    if v < 0: suf = 0             # đặt lại: chỉ giữ phần sau phần tử xấu cuối cùng
    else: suf += v
print("tiền tố đến điểm dừng:", pref, "| hậu tố sau điểm xấu cuối:", suf)

tiền tố đến điểm dừng: 7 | hậu tố sau điểm xấu cuối: 11


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Tính tay ví dụ 2, `root = [1,2,3,4,5,null,7]`:

```
        1
      /   \
     2     3
    / \     \
   4   5     7
```

| tầng | chiều | thứ tự duyệt | điều kiện | được cộng | tổng |
|---|---|---|---|---|---|
| 1 | trái→phải | 1 | có con trái? 1 có (2) | 1 | 1 |
| 2 | phải→trái | 3, 2 | có con phải? 3 có (7), 2 có (5) | 3, 2 | 5 |
| 3 | trái→phải | 4, 5, 7 | có con trái? 4 **không** ⇒ dừng | – | 0 |

Đáp án `[1, 5, 0]`. Chú ý: ở tầng cuối (lá) điều kiện luôn thất bại ngay nút đầu ⇒ tổng 0, nhưng vẫn phải có phần tử trong `ans`.

## 5. Cách 1 — Baseline: lấy từng tầng bằng cách đi lại từ gốc

**Ý tưởng:** với mỗi độ sâu `d = 1, 2, …`, DFS từ gốc để thu các nút ở độ sâu `d` (theo thứ tự trái → phải), rồi áp dụng quy tắc dừng. Dừng khi không còn nút ở độ sâu `d`.

DFS dùng ngăn xếp (tránh tràn đệ quy với cây lệch 10^5 tầng).

In [6]:
def nodes_at_depth(root, d):
    res = []
    stack = [(root, 1)]
    while stack:
        node, depth = stack.pop()
        if depth == d:
            res.append(node)
            continue
        # đẩy phải trước để lấy trái trước (thứ tự trái -> phải)
        if node.right: stack.append((node.right, depth + 1))
        if node.left: stack.append((node.left, depth + 1))
    return res

def level_sum(level, odd):
    s = 0
    seq = level if odd else reversed(level)
    for node in seq:
        if (node.left if odd else node.right) is None:
            break
        s += node.val
    return s

def baseline(root):
    ans, d = [], 1
    while True:
        level = nodes_at_depth(root, d)
        if not level:
            return ans
        ans.append(level_sum(level, d % 2 == 1))
        d += 1

for arr, exp in examples:
    print(baseline(build(arr)), "kỳ vọng", exp)

[5, 8, 0] kỳ vọng [5, 8, 0]
[1, 5, 0] kỳ vọng [1, 5, 0]


**Độ phức tạp:** mỗi độ sâu lại duyệt toàn bộ phần cây phía trên ⇒ `O(n · H)` với `H` là chiều cao. Cây lệch (`H = n = 10^5`) cho `10^{10}` bước.

**Nút thắt:** thu từng tầng **riêng rẽ** từ gốc. BFS cho mọi tầng trong **một lượt**.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — BFS cho sẵn từng tầng theo thứ tự trái → phải.** Ta chỉ cần với mỗi tầng chọn chiều duyệt theo cờ `left_to_right` (đổi sau mỗi tầng) và cộng đến điểm dừng.

**Insight 2 — Một lượt DFS cũng đủ.** Nếu DFS theo thứ tự "trái trước" thì các nút cùng tầng xuất hiện theo thứ tự trái → phải. Khi đó:

- Tầng lẻ (trái → phải, dừng ở nút thiếu con trái đầu tiên) = **tổng tiền tố** tới điểm dừng: giữ cờ `stopped[d]`.
- Tầng chẵn (phải → trái, dừng ở nút thiếu con phải đầu tiên **tính từ phải**) = **tổng các nút nằm sau nút thiếu con phải cuối cùng** khi đọc trái → phải: mỗi khi gặp nút thiếu con phải thì **đặt lại** tổng tầng đó về 0.

**Quy tắc "nếu … thì …":**

- Tầng lẻ: nếu đã dừng thì bỏ qua; nếu nút không có con trái thì đánh dấu dừng; ngược lại cộng.
- Tầng chẵn: nếu nút không có con phải thì `sum[d] = 0`; ngược lại `sum[d] += val`.

## 7. Các cách tiếp theo

- **Cách 2:** BFS theo tầng + cờ chiều (lời giải nộp chính, `O(n)`).
- **Cách 3:** DFS gom nút theo độ sâu vào `levels[d]`, rồi xử lý như Cách 2.
- **Cách 4:** DFS **một lượt** tính trực tiếp bằng tiền tố (tầng lẻ) / hậu tố có đặt lại (tầng chẵn) — không lưu cả tầng.

### 7.1. Cách 2 — BFS theo tầng + cờ chiều

**Giải thích từng đoạn:**

1. `q = [root]`, `left = True` (tầng 1 duyệt trái → phải).
2. Mỗi vòng: dựng `nq` (tầng kế tiếp) từ `q`; duyệt `q` theo chiều `left`, kiểm tra con tương ứng, `break` khi thiếu, cộng dồn `s`.
3. `ans.append(s)`, đảo `left`, `q = nq`.

In [7]:
def bfs_solution(root):
    if root is None:
        return []
    q, ans, left = [root], [], True
    while q:
        nq = []
        for node in q:
            if node.left: nq.append(node.left)
            if node.right: nq.append(node.right)
        s = 0
        m = len(q)
        for i in range(m):
            node = q[i] if left else q[m - 1 - i]
            child = node.left if left else node.right
            if child is None:
                break
            s += node.val
        ans.append(s)
        left = not left
        q = nq
    return ans

for arr, exp in examples:
    print(bfs_solution(build(arr)), "kỳ vọng", exp)

[5, 8, 0] kỳ vọng [5, 8, 0]
[1, 5, 0] kỳ vọng [1, 5, 0]


**Độ phức tạp:** mỗi nút vào `nq` đúng một lần và được xét tối đa một lần trong vòng cộng ⇒ `O(n)` thời gian, `O(W)` bộ nhớ (`W` = độ rộng lớn nhất, ≤ `n`).

### 7.2. Cách 3 — DFS gom theo độ sâu

Dùng ngăn xếp với `(node, depth)`, đẩy con phải trước con trái ⇒ lấy ra theo thứ tự tiền thứ tự trái trước ⇒ trong mỗi `levels[d]` các nút theo thứ tự trái → phải.

In [8]:
def dfs_collect(root):
    levels = []
    stack = [(root, 0)]
    while stack:
        node, d = stack.pop()
        if d == len(levels):
            levels.append([])
        levels[d].append(node)
        if node.right: stack.append((node.right, d + 1))
        if node.left: stack.append((node.left, d + 1))
    return [level_sum(levels[d], d % 2 == 0) for d in range(len(levels))]

for arr, exp in examples:
    print(dfs_collect(build(arr)), "kỳ vọng", exp)

[5, 8, 0] kỳ vọng [5, 8, 0]
[1, 5, 0] kỳ vọng [1, 5, 0]


**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ (lưu mọi nút theo tầng).

### 7.3. Cách 4 — DFS một lượt: tiền tố cho tầng lẻ, hậu tố có đặt lại cho tầng chẵn

**Giải thích:** DFS trái trước (ngăn xếp). Với nút ở độ sâu `d` (0-indexed; tầng lẻ 1-indexed ⇔ `d` chẵn):

- `d` chẵn (tầng lẻ): nếu `stopped[d]` thì bỏ qua; nếu `node.left is None` thì `stopped[d] = True`; ngược lại `sums[d] += val`.
- `d` lẻ (tầng chẵn): nếu `node.right is None` thì `sums[d] = 0`; ngược lại `sums[d] += val`.

Không cần lưu cả tầng ⇒ bộ nhớ chỉ là ngăn xếp DFS + `O(H)` cho các mảng theo độ sâu.

In [9]:
def dfs_one_pass(root):
    sums, stopped = [], []
    stack = [(root, 0)]
    while stack:
        node, d = stack.pop()
        if d == len(sums):
            sums.append(0); stopped.append(False)
        if d % 2 == 0:                       # tầng lẻ (1-indexed): trái -> phải
            if not stopped[d]:
                if node.left is None:
                    stopped[d] = True
                else:
                    sums[d] += node.val
        else:                                # tầng chẵn: phải -> trái
            if node.right is None:
                sums[d] = 0
            else:
                sums[d] += node.val
        if node.right: stack.append((node.right, d + 1))
        if node.left: stack.append((node.left, d + 1))
    return sums

for arr, exp in examples:
    print(dfs_one_pass(build(arr)), "kỳ vọng", exp)

[5, 8, 0] kỳ vọng [5, 8, 0]
[1, 5, 0] kỳ vọng [1, 5, 0]


In [10]:
class SolutionDFS:
    def zigzagLevelSum(self, root: Optional[TreeNode]) -> List[int]:
        return dfs_one_pass(root)

print([SolutionDFS().zigzagLevelSum(build(a)) for a, _ in examples])

[[5, 8, 0], [1, 5, 0]]


**Độ phức tạp:** `O(n)` thời gian, `O(H)` bộ nhớ cho `sums/stopped` + ngăn xếp DFS.

## 8. TRACE từng bước

Trace Cách 2 trên ví dụ 1, `root = [5,2,8,1,null,9,6]`:

| tầng | q (trái→phải) | chiều | nút xét | con cần có | kết quả | tổng |
|---|---|---|---|---|---|---|
| 1 | [5] | → | 5 | trái = 2 ✔ | cộng 5 | 5 |
| 2 | [2, 8] | ← | 8 | phải = 6 ✔ | cộng 8 | |
| | | | 2 | phải = None ✘ | dừng | 8 |
| 3 | [1, 9, 6] | → | 1 | trái = None ✘ | dừng | 0 |

In [11]:
def trace_bfs(root):
    q, left, level = [root], True, 1
    ans = []
    while q:
        nq = [c for n in q for c in (n.left, n.right) if c]
        order = q if left else q[::-1]
        print(f"tầng {level} ({'trái->phải' if left else 'phải->trái'}), q = {[n.val for n in q]}")
        s = 0
        for node in order:
            child = node.left if left else node.right
            if child is None:
                print(f"   nút {node.val}: thiếu con {'trái' if left else 'phải'} -> dừng")
                break
            s += node.val
            print(f"   nút {node.val}: có con {'trái' if left else 'phải'} ({child.val}) -> cộng, s = {s}")
        ans.append(s)
        left, q, level = not left, nq, level + 1
    print("ans =", ans)

trace_bfs(build([5, 2, 8, 1, None, 9, 6]))

tầng 1 (trái->phải), q = [5]
   nút 5: có con trái (2) -> cộng, s = 5
tầng 2 (phải->trái), q = [2, 8]
   nút 8: có con phải (6) -> cộng, s = 8
   nút 2: thiếu con phải -> dừng
tầng 3 (trái->phải), q = [1, 9, 6]
   nút 1: thiếu con trái -> dừng
ans = [5, 8, 0]


In [12]:
# Trace Cách 4 (DFS một lượt) trên cây có tầng chẵn với nút thiếu con phải ở giữa
#         1
#      /     \
#     2       3
#    / \     /
#   4   5   6
#  tầng 2 (phải->trái): 3 thiếu con phải -> dừng ngay -> 0
arr = [1, 2, 3, 4, 5, 6]
root = build(arr)
sums, stopped = [], []
stack = [(root, 0)]
while stack:
    node, d = stack.pop()
    if d == len(sums): sums.append(0); stopped.append(False)
    before = sums[d]
    if d % 2 == 0:
        if not stopped[d]:
            if node.left is None: stopped[d] = True
            else: sums[d] += node.val
    else:
        if node.right is None: sums[d] = 0
        else: sums[d] += node.val
    print(f"nút {node.val} ở độ sâu {d}: sums[{d}] {before} -> {sums[d]}, stopped={stopped[d]}")
    if node.right: stack.append((node.right, d + 1))
    if node.left: stack.append((node.left, d + 1))
print("kết quả:", sums, "| BFS:", bfs_solution(root))

nút 1 ở độ sâu 0: sums[0] 0 -> 1, stopped=False
nút 2 ở độ sâu 1: sums[1] 0 -> 2, stopped=False
nút 4 ở độ sâu 2: sums[2] 0 -> 0, stopped=True
nút 5 ở độ sâu 2: sums[2] 0 -> 0, stopped=True
nút 3 ở độ sâu 1: sums[1] 2 -> 0, stopped=False
nút 6 ở độ sâu 2: sums[2] 0 -> 0, stopped=True
kết quả: [1, 0, 0] | BFS: [1, 0, 0]


## 9. Code nộp LeetCode

In [13]:
class Solution:
    def zigzagLevelSum(self, root: Optional[TreeNode]) -> List[int]:
        ans = []
        if root is None:
            return ans
        q = [root]
        left = True
        while q:
            nq = []
            for node in q:
                if node.left:
                    nq.append(node.left)
                if node.right:
                    nq.append(node.right)
            s = 0
            m = len(q)
            for i in range(m):
                node = q[i] if left else q[m - 1 - i]
                if (node.left if left else node.right) is None:
                    break
                s += node.val
            ans.append(s)
            left = not left
            q = nq
        return ans

sol = Solution()
for arr, exp in examples:
    got = sol.zigzagLevelSum(build(arr))
    print(got, "✔" if got == exp else "✘")

[5, 8, 0] ✔
[1, 5, 0] ✔


## 10. Vì sao thuật toán đúng

1. **BFS đúng tầng, đúng thứ tự:** nếu `q` là tầng `t` theo thứ tự trái → phải, thì `nq` (con trái rồi con phải của từng nút theo thứ tự) là tầng `t+1` theo thứ tự trái → phải. Quy nạp từ `q = [root]`.
2. **Chiều duyệt:** cờ `left` bắt đầu `True` ở tầng 1 và đổi sau mỗi tầng ⇒ tầng lẻ trái → phải, tầng chẵn phải → trái.
3. **Điểm dừng:** vòng `for` cộng đúng các nút đứng trước nút vi phạm đầu tiên theo chiều đang xét và `break` tại đó — đúng định nghĩa.
4. **Cách 4:** với tầng chẵn, các nút được cộng (khi đọc phải → trái, trước nút thiếu con phải đầu tiên) chính là các nút **bên phải** nút thiếu con phải **cuối cùng** khi đọc trái → phải. Việc đặt lại tổng về 0 tại mỗi nút vi phạm giữ đúng bất biến "`sums[d]` = tổng các nút đã gặp sau nút vi phạm gần nhất".

## 11. Phân tích Time / Space Complexity

Đặt `n` = số nút, `H` = chiều cao, `W` = độ rộng lớn nhất.

| Cách | Time | Space |
|---|---|---|
| 1. Đi lại từ gốc cho từng tầng | `O(n · H)` | `O(H)` |
| 2. BFS + cờ chiều | `O(n)` | `O(W)` |
| 3. DFS gom theo độ sâu | `O(n)` | `O(n)` |
| 4. DFS một lượt | `O(n)` | `O(H)` |

Cây đầy đủ: `W ≈ n/2`, `H ≈ log n` ⇒ Cách 4 tiết kiệm bộ nhớ. Cây lệch: `H = n`, `W = 1` ⇒ Cách 2 tiết kiệm hơn.

## 12. Bộ test

In [14]:
def run_all(arr):
    root = build(arr)
    res = [bfs_solution(root), dfs_collect(root), dfs_one_pass(root), Solution().zigzagLevelSum(root)]
    if len(arr) <= 3000:
        res.append(baseline(root))
    return res

for arr, exp in examples:
    assert all(r == exp for r in run_all(arr)), run_all(arr)
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [15]:
# 12.2 ca biên (tự dựng, đối chiếu lại đề gốc nếu có Premium)
cases = [
    ([7], [0]),                                  # 1 nút: thiếu con trái -> 0
    ([-3, 1], [-3, 0]),                          # gốc có con trái, giá trị âm vẫn cộng
    ([1, None, 2], [0, 0]),                      # gốc thiếu con trái -> 0; tầng 2: nút 2 thiếu con phải -> 0
    ([1, 2, 3, None, 4, None, 5], [1, 5, 0]),    # tầng 2: 3 (phải 5) ✔, 2 (phải 4) ✔
    ([1, 2, 3, 4, None, 5, 6], [1, 3, 0]),       # tầng 2: 3 ✔, 2 thiếu phải -> dừng
]
for arr, exp in cases:
    r = run_all(arr)
    assert all(x == exp for x in r), (arr, r)
print("Ca biên: OK")

Ca biên: OK


In [16]:
# 12.3 phản ví dụ: hiểu nhầm "bỏ qua nút vi phạm rồi đi tiếp" thay vì dừng
def wrong_skip(root):
    q, ans, left = [root], [], True
    while q:
        nq = [c for n in q for c in (n.left, n.right) if c]
        seq = q if left else q[::-1]
        ans.append(sum(n.val for n in seq if (n.left if left else n.right)))   # SAI: không dừng
        left, q = not left, nq
    return ans

root = build([1, 2, 3, None, 4, 5])     # tầng 2 (phải->trái): 3 thiếu con phải -> dừng; 2 có con phải
print("sai (bỏ qua):", wrong_skip(root), "| đúng (dừng):", Solution().zigzagLevelSum(root))

sai (bỏ qua): [1, 2, 0] | đúng (dừng): [1, 0, 0]


In [17]:
# 12.4 đối chiếu ngẫu nhiên trên cây ngẫu nhiên
def random_tree_array(rng, n):
    arr = [rng.randint(-20, 20)]
    count = 1
    i = 0
    while count < n:
        if arr[i] is not None:
            for _ in range(2):
                if count < n and rng.random() < 0.75:
                    arr.append(rng.randint(-20, 20)); count += 1
                else:
                    arr.append(None)
        i += 1
        if i >= len(arr):
            break
    return arr

rng = random.Random(3902)
for _ in range(400):
    arr = random_tree_array(rng, rng.randint(1, 40))
    r = run_all(arr)
    assert all(x == r[0] for x in r), (arr, r)
print("400 cây ngẫu nhiên: OK")

400 cây ngẫu nhiên: OK


In [18]:
# 12.5 cây lớn: cây lệch 10^5 tầng và cây đầy đủ ~10^5 nút
n = 10**5
root = TreeNode(1); cur = root
for i in range(2, n + 1):
    cur.left = TreeNode(i); cur = cur.left          # lệch trái
t0 = time.time(); a = Solution().zigzagLevelSum(root); t1 = time.time()
b = dfs_one_pass(root)
print("cây lệch: số tầng", len(a), "| khớp DFS:", a == b, "| thời gian < 1s:", t1 - t0 < 1)
arr = [rng.randint(-10**5, 10**5) for _ in range(n)]
root = build(arr)
print("cây đầy đủ: khớp 3 cách:", bfs_solution(root) == dfs_one_pass(root) == dfs_collect(root))

cây lệch: số tầng 100000 | khớp DFS: True | thời gian < 1s: True


cây đầy đủ: khớp 3 cách: True


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Bỏ qua nút vi phạm rồi tiếp tục cộng

Đã minh họa ở 12.3. Đề yêu cầu **dừng ngay**, không phải lọc. **Sửa:** `break`.

### Lỗi 2 — Nhầm điều kiện: tầng chẵn kiểm tra con trái

In [19]:
def wrong_condition(root):
    q, ans, left = [root], [], True
    while q:
        nq = [c for n in q for c in (n.left, n.right) if c]
        s = 0
        for node in (q if left else q[::-1]):
            if node.left is None:                 # SAI: tầng chẵn phải kiểm tra con PHẢI
                break
            s += node.val
        ans.append(s); left, q = not left, nq
    return ans

root = build([5, 2, 8, 1, None, 9, 6])
print("sai:", wrong_condition(root), "| đúng:", Solution().zigzagLevelSum(root))

sai: [5, 10, 0] | đúng: [5, 8, 0]


### Lỗi 3 — Đảo chiều cả hàng đợi rồi dùng nó để dựng tầng sau

Nếu đảo `q` tại chỗ (`q.reverse()`) để duyệt phải → trái, rồi mới dựng `nq` từ `q` đã đảo, thứ tự tầng sau bị đảo theo ⇒ các tầng sau sai chiều.

In [20]:
def wrong_reverse_inplace(root):
    q, ans, left = [root], [], True
    while q:
        if not left:
            q.reverse()                            # SAI: làm hỏng thứ tự trái -> phải
        s = 0
        for node in q:
            if (node.left if left else node.right) is None: break
            s += node.val
        ans.append(s)
        q = [c for n in q for c in (n.left, n.right) if c]
        left = not left
    return ans

root = build([1, 2, 3, 4, 5, 6, 7, None, None, 8, 9, 10, 11, 12, 13])   # nút 4 là lá
print("sai:", wrong_reverse_inplace(root), "| đúng:", Solution().zigzagLevelSum(root))

sai: [1, 5, 13, 0] | đúng: [1, 5, 0, 0]


**Sửa:** giữ `q` luôn theo thứ tự trái → phải; chỉ **đọc** ngược bằng chỉ số hoặc `reversed`.

### Lỗi 4 — Quên tầng cuối (hoặc tầng có tổng 0)

Một số cài đặt chỉ `append` khi `s > 0`, hoặc dừng vòng lặp khi tầng không đóng góp — nhưng đề cần **một phần tử cho mỗi tầng**.

In [21]:
def wrong_skip_zero(root):
    return [s for s in Solution().zigzagLevelSum(root) if s != 0]      # SAI

root = build([5, 2, 8, 1, None, 9, 6])
print("sai:", wrong_skip_zero(root), "| đúng:", Solution().zigzagLevelSum(root))

sai: [5, 8] | đúng: [5, 8, 0]


### Lỗi 5 — DFS đệ quy bị tràn ngăn xếp với cây lệch 10^5 tầng

Python mặc định giới hạn đệ quy ~1000. Với cây lệch, DFS đệ quy gây `RecursionError`. **Sửa:** dùng BFS hoặc DFS bằng ngăn xếp tự quản lý (như Cách 3, 4).

In [22]:
import sys
print("giới hạn đệ quy mặc định:", sys.getrecursionlimit(), "< 10^5 tầng có thể có")

giới hạn đệ quy mặc định: 1000 < 10^5 tầng có thể có


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Theo tầng" + "zigzag"** ⇒ BFS giữ tầng theo thứ tự trái → phải, chỉ đổi **chiều đọc** (giống LC 103 *Binary Tree Zigzag Level Order Traversal*).
- **"Dừng ở phần tử vi phạm đầu tiên"** ⇒ vòng lặp có `break`; khi phải làm ngược chiều mà dữ liệu đến xuôi chiều ⇒ "đặt lại tổng" tại mỗi phần tử vi phạm.
- Cây tới `10^5` nút ⇒ tránh đệ quy sâu.
- Tầng lá: điều kiện "có con" luôn sai ⇒ tổng tầng cuối luôn `0`.

## 15. Biến thể của bài

1. **Không dừng, chỉ lọc** (cộng mọi nút thỏa điều kiện): bỏ `break`.
2. **Trả về danh sách nút được duyệt** thay vì tổng: append `node.val` thay vì cộng.
3. **Chiều bắt đầu từ phải** hoặc điều kiện ngược (tầng lẻ cần con phải…): đổi cờ khởi tạo / thuộc tính kiểm tra.
4. **Trả về tầng có tổng lớn nhất** (giống LC 1161 *Maximum Level Sum of a Binary Tree*).
5. Liên quan: LC 102 *Level Order Traversal*, LC 103 *Zigzag Level Order*, LC 199 *Right Side View*.

In [23]:
# Biến thể 2: danh sách giá trị được duyệt ở mỗi tầng
def zigzag_processed(root):
    q, out, left = [root], [], True
    while q:
        nq = [c for n in q for c in (n.left, n.right) if c]
        taken = []
        for node in (q if left else reversed(q)):
            if (node.left if left else node.right) is None: break
            taken.append(node.val)
        out.append(taken); left, q = not left, nq
    return out

root = build([1, 2, 3, 4, 5, None, 7])
print(zigzag_processed(root), "-> tổng:", [sum(t) for t in zigzag_processed(root)])

[[1], [3, 2], []] -> tổng: [1, 5, 0]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Đi lại từ gốc | Thu tầng `d` bằng DFS riêng | `O(n H)` | `O(H)` | Chỉ để hiểu/kiểm thử |
| 2. BFS + cờ chiều | Tầng trái→phải, đọc theo chiều | `O(n)` | `O(W)` | **Lời giải chuẩn** |
| 3. DFS gom tầng | `levels[d]` rồi xử lý | `O(n)` | `O(n)` | Khi đã có sẵn khung DFS |
| 4. DFS một lượt | Tiền tố / hậu tố có đặt lại | `O(n)` | `O(H)` | Tiết kiệm bộ nhớ với cây rộng |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `root = [1, 2, 3]` cho kết quả gì?

**Bài 2 (tính tay).** `root = [10, 5, 15, 3, 7, 12, 20, 1, None, 6]` — vẽ cây và tính `ans`.

**Bài 3 (điền TODO).** Viết `zigzag_count(root)` trả về **số nút được duyệt** ở mỗi tầng.

**Bài 4 (tự viết lại).** Viết lại Cách 2 bằng `collections.deque` và `len(q)` để tách tầng (không dùng `nq`).

**Bài 5 (mini challenge).** Biến thể: tầng lẻ dừng ở nút **không có đủ hai con**, tầng chẵn dừng ở nút **là lá**. Viết và kiểm chứng với cài đặt "thu tầng rồi xử lý".

In [24]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def zigzag_count(root):
    # TODO: như Cách 2 nhưng đếm số nút thay vì cộng giá trị
    pass

In [25]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def zigzag_variant(root):
    # TODO: điều kiện dừng mới cho tầng lẻ / chẵn
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Tầng 1: gốc có con trái ⇒ 1. (2) Tầng 2 phải → trái: 3 là lá ⇒ dừng ngay. (3) `[1, 0]`.

**Bài 2.** (1) Tầng 2 = `[5, 15]`, tầng 3 = `[3, 7, 12, 20]`, tầng 4 = `[1, 6]`. (2) Tầng 2 (←): 15 có con phải 20 ✔, 5 có con phải 7 ✔ ⇒ 20. Tầng 3 (→): 3 có con trái 1 ✔, 7 có con trái 6 ✔, 12 thiếu ⇒ dừng ⇒ 10. (3) `[10, 20, 10, 0]`.

**Bài 3.** (1) Thay `s += node.val` bằng `s += 1`. (2) Giữ `break`. (3) Tầng cuối luôn 0.

**Bài 4.** (1) `for _ in range(len(q))` lấy đúng một tầng. (2) Cần lưu tầng ra list tạm để đọc ngược. (3) So kết quả với `Solution`.

**Bài 5.** (1) Tầng lẻ: `node.left and node.right`. (2) Tầng chẵn: `node.left or node.right`. (3) Viết bản "thu tầng" để đối chiếu.

## 19. Đáp án tham khảo cho bài tập

In [26]:
# Bài 1, 2
assert Solution().zigzagLevelSum(build([1, 2, 3])) == [1, 0]
assert Solution().zigzagLevelSum(build([10, 5, 15, 3, 7, 12, 20, 1, None, 6])) == [10, 20, 10, 0]
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [27]:
# Bài 3
def zigzag_count_sol(root):
    q, out, left = [root], [], True
    while q:
        nq = [c for n in q for c in (n.left, n.right) if c]
        c = 0
        for node in (q if left else reversed(q)):
            if (node.left if left else node.right) is None: break
            c += 1
        out.append(c); left, q = not left, nq
    return out

assert zigzag_count_sol(build([1, 2, 3, 4, 5, None, 7])) == [1, 2, 0]
print("Bài 3 OK")

Bài 3 OK


In [28]:
# Bài 4
def sol_deque(root):
    if root is None: return []
    dq, ans, left = deque([root]), [], True
    while dq:
        level = [dq.popleft() for _ in range(len(dq))]
        for node in level:
            if node.left: dq.append(node.left)
            if node.right: dq.append(node.right)
        s = 0
        for node in (level if left else reversed(level)):
            if (node.left if left else node.right) is None: break
            s += node.val
        ans.append(s); left = not left
    return ans

rng = random.Random(4)
for _ in range(200):
    root = build(random_tree_array(rng, rng.randint(1, 30)))
    assert sol_deque(root) == Solution().zigzagLevelSum(root)
print("Bài 4 OK")

Bài 4 OK


In [29]:
# Bài 5
def zigzag_variant_sol(root):
    q, out, left = [root], [], True
    while q:
        nq = [c for n in q for c in (n.left, n.right) if c]
        s = 0
        for node in (q if left else reversed(q)):
            ok = (node.left and node.right) if left else (node.left or node.right)
            if not ok: break
            s += node.val
        out.append(s); left, q = not left, nq
    return out

def zigzag_variant_ref(root):
    levels = levels_nodes = []
    cur = [root]
    while cur:
        levels.append(cur); cur = [c for n in cur for c in (n.left, n.right) if c]
    out = []
    for d, lv in enumerate(levels):
        seq = lv if d % 2 == 0 else lv[::-1]
        s = 0
        for node in seq:
            if d % 2 == 0 and not (node.left and node.right): break
            if d % 2 == 1 and not (node.left or node.right): break
            s += node.val
        out.append(s)
    return out

rng = random.Random(5)
for _ in range(200):
    root = build(random_tree_array(rng, rng.randint(1, 30)))
    assert zigzag_variant_sol(root) == zigzag_variant_ref(root)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
q, ans, left = [root], [], True
while q:
    nq = [c for n in q for c in (n.left, n.right) if c]   # tầng sau, trái -> phải
    s = 0
    for node in (q if left else reversed(q)):             # chỉ đổi chiều ĐỌC
        if (node.left if left else node.right) is None:
            break                                          # dừng ngay
        s += node.val
    ans.append(s)                                          # kể cả s = 0
    left, q = not left, nq
```

**Checklist:**

- [ ] Tầng lẻ trái → phải kiểm tra **con trái**; tầng chẵn phải → trái kiểm tra **con phải**?
- [ ] `break` ngay tại nút vi phạm (không bỏ qua rồi đi tiếp)?
- [ ] Không đảo `q` tại chỗ; tầng sau luôn dựng theo thứ tự trái → phải?
- [ ] Có phần tử cho **mọi** tầng, kể cả tổng `0`?
- [ ] Không dùng đệ quy sâu (cây lệch `10^5` tầng)?